# Montoya VRP-REP benchmark: conditional real-instance demo

This notebook uses a **real XML file only if you manually downloaded one** from https://www.vrp-rep.org/datasets/item/2016-0020.html into `data/raw/montoya`. No original instance is shipped; in its absence the notebook explicitly reports **[not run]** and does not substitute fake benchmark data. The route-construction choice and calibration are ours, not the unspecified original code.

In [ ]:
from pathlib import Path
from befrg import read_montoya_xml, make_problem, BEFRGSolver, validate_solution
from befrg.route_init import cheapest_insertion
# Notebook launched from repo root (recommended).
folder=Path("data/raw/montoya")
files=sorted(folder.rglob("*.xml"))
preferred=[p for p in files if p.stem=="tc0c20s3cf2"]
xml=(preferred or files[:1])[0] if files else None
print("[original benchmark file]", xml if xml else "[not run] Montoya XML not found")

## Parse benchmark and construct a fixed route
Only run the following cell when `xml` exists. We use a prefix for a modest demonstration. The paper does not supply an exact route sequence or sufficient energy-model calibration to claim numerical table reproduction.

In [ ]:
if xml is None:
    print("[not run] Download a Montoya XML file first; see data/README.md")
else:
    raw=read_montoya_xml(xml)
    order=cheapest_insertion(raw)[:min(10,len(raw.customers))]
    # Assumed empty vehicle mass 1000 kg; theta chosen to match 0.125 kWh/km empty rate.
    problem=make_problem(raw,order,gamma=16,Gamma=16,veh_mass=1000.)
    print("Original dataset:",raw.name,"available stations:",len(problem.stations),"tour customers:",len(order))
    print("Assumed vehicle mass",problem.veh_mass,"theta",problem.theta)

## Run original BEFRG and independent replay
The floored DP may return a plan that fails continuous physical charging validation. Such a failure is a research finding, not a reason to suppress the case.

In [ ]:
if xml is not None:
    try:
        result=BEFRGSolver(problem).bottom_up()
        replay=validate_solution(problem,result)
        print("[ours-full-input, not exact paper reproduction] DP model time",result.total_minutes)
        print("[ours-full-input] physical feasible",replay.feasible,"errors",replay.errors)
        print("Detours",result.detours,"physical return SoC",replay.final_soc_kwh)
    except ValueError as exc:
        print("[not run] No feasible positive-state plan for chosen route:",exc)